# Course 1 — Linear Regression

Live-coding notebook mirroring the slides.
Part 1: simple LR on the ISLP `Boston` dataset (`medv` ~ `lstat`).
Part 2: multiple LR, diagnostics, and categorical predictors.

**Part 1 sections**
1. Fitting a line (0:00–0:30)
2. Reading the output (0:30–1:00)
3. Inference for β (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
rng = np.random.default_rng(0)
boston = load_dataset('boston')
boston.head()


## 1. Fitting a line

### The dataset

In [ ]:
print(boston.shape)
print(boston.columns.tolist())
print(boston[['lstat', 'medv']].describe().round(2))


### Whiteboard formula — least squares

$$\hat\beta_1 = \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sum_i (x_i - \bar x)^2}, \qquad \hat\beta_0 = \bar y - \hat\beta_1 \bar x$$

In [ ]:
x = boston['lstat'].to_numpy()
y = boston['medv'].to_numpy()
x_bar, y_bar = x.mean(), y.mean()
b1 = ((x - x_bar) * (y - y_bar)).sum() / ((x - x_bar)**2).sum()
b0 = y_bar - b1 * x_bar
print(f'by hand: intercept={b0:.4f}, slope={b1:.4f}')


### Same fit, three lines of sklearn

In [ ]:
X = boston[['lstat']]
model = LinearRegression().fit(X, y)
print(f'sklearn: intercept={model.intercept_:.4f}, slope={model.coef_[0]:.4f}')


### Plot the data and the fit

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x, y, s=10, alpha=0.6)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, b0 + b1 * x_line, color='C3', linewidth=2)
ax.set_xlabel('lstat (% lower status)'); ax.set_ylabel('medv ($1000s)')
ax.set_title('Boston: medv ~ lstat'); plt.show()


## 2. Reading the output

### Predictions and residuals

In [ ]:
y_hat = model.predict(X)
resid = y - y_hat
print('first 5 predictions:', y_hat[:5].round(2))
print('first 5 residuals: ', resid[:5].round(2))


### R² and RMSE

In [ ]:
r2 = r2_score(y, y_hat)
rmse = np.sqrt(mean_squared_error(y, y_hat))
print(f'R^2  = {r2:.4f}   (fraction of variance explained)')
print(f'RMSE = {rmse:.4f} (in same units as medv: $1000s)')


### Residual plot — your first diagnostic

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(y_hat, resid, s=10, alpha=0.6)
ax.axhline(0, color='C3', linewidth=1)
ax.set_xlabel('fitted'); ax.set_ylabel('residual')
ax.set_title('Residuals vs fitted — curvature is a red flag'); plt.show()


The clear U-shape says: a *line* is the wrong shape for medv ~ lstat.
We will fix this next week with polynomial features.

## 3. Inference for β

### Standard error and t-statistic

In [ ]:
n, p = len(y), 1
rss = (resid**2).sum()
sigma2 = rss / (n - p - 1)
se_b1 = np.sqrt(sigma2 / ((x - x_bar)**2).sum())
t = b1 / se_b1
print(f'SE(beta1) = {se_b1:.4f}')
print(f't-stat    = {t:.2f}   (huge -> reject beta1 = 0)')


### Bootstrap CI for the slope

In [ ]:
n_boot = 2000
b1_boot = np.empty(n_boot)
idx = np.arange(n)
for k in range(n_boot):
    sample = rng.choice(idx, n, replace=True)
    xb, yb = x[sample], y[sample]
    b1_boot[k] = ((xb - xb.mean()) * (yb - yb.mean())).sum() / ((xb - xb.mean())**2).sum()
lo, hi = np.quantile(b1_boot, [0.025, 0.975])
print(f'95% bootstrap CI for slope: [{lo:.4f}, {hi:.4f}]')


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(b1_boot, bins=40, color='C0', alpha=0.7)
ax.axvline(b1, color='C3', label='point estimate')
ax.set_xlabel('bootstrapped slope'); ax.legend(); plt.show()


### Four assumptions, four diagnostics

1. **Linearity** — residuals vs fitted should look like noise.
2. **Constant variance** — same plot, no funnel.
3. **Independence** — no serial correlation (matters for time-series data).
4. **Normality** — Q-Q plot of residuals.

In [ ]:
from scipy import stats
fig, ax = plt.subplots(figsize=(5, 4))
stats.probplot(resid, plot=ax)
ax.set_title('Q-Q plot of residuals'); plt.show()


### Recap
- Least squares = minimize Σ residuals². The closed form is two lines of arithmetic.
- R² and RMSE summarize fit. Residual plots find what they miss.
- A confidence interval is what a single coefficient is worth as a *claim*.
- The bootstrap gives a CI for *anything* without distributional assumptions.

---

# Part 2 — Multiple LR & diagnostics

Multiple regression, diagnostics, qualitative predictors and
interactions. We continue with `Boston` and bring in `Carseats`.

**Part 2 sections**
4. Multiple LR and collinearity (1:30–2:00)
5. Diagnostic plots and leverage (2:00–2:30)
6. Qualitative predictors and interactions (2:30–3:00)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
boston = load_dataset('boston')
carseats = load_dataset('carseats')
print('boston   :', boston.shape)
print('carseats :', carseats.shape)


## 1. Multiple LR and collinearity

### Fit `medv` on all 12 predictors

In [ ]:
X = boston.drop(columns=['medv'])
y = boston['medv']
m = LinearRegression().fit(X, y)
coefs = pd.Series(m.coef_, index=X.columns).round(4)
print(coefs.to_string())
print(f'R^2 = {m.score(X, y):.4f}')


### Variance Inflation Factor — VIF

For predictor j, fit `x_j ~ rest`, get R²_j, then VIF_j = 1/(1 - R²_j).
Anything above 5–10 is collinear with the rest of the design.

In [ ]:
def vif(X: pd.DataFrame) -> pd.Series:
    out = {}
    for col in X.columns:
        others = X.drop(columns=[col])
        r2 = LinearRegression().fit(others, X[col]).score(others, X[col])
        out[col] = 1.0 / (1.0 - r2)
    return pd.Series(out).sort_values(ascending=False)

print(vif(X).round(2).to_string())


`tax`, `rad`, `nox` are the biggest offenders. Coefficient *signs* on a
regression with collinear predictors can flip when you add or drop a
feature — never read coefficients in isolation.

## 2. Diagnostic plots and leverage

In [ ]:
y_hat = m.predict(X)
resid = y - y_hat
std_resid = resid / resid.std()

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes[0, 0].scatter(y_hat, resid, s=8, alpha=0.6)
axes[0, 0].axhline(0, color='C3'); axes[0, 0].set_title('Residuals vs Fitted')
axes[0, 0].set_xlabel('fitted'); axes[0, 0].set_ylabel('residual')

from scipy import stats
stats.probplot(resid, plot=axes[0, 1])
axes[0, 1].set_title('Q-Q')

axes[1, 0].scatter(y_hat, np.sqrt(np.abs(std_resid)), s=8, alpha=0.6)
axes[1, 0].set_title('Scale-Location'); axes[1, 0].set_xlabel('fitted')
axes[1, 0].set_ylabel(r'$\sqrt{|standardized\ resid|}$')

# Leverage: diag of the hat matrix H = X(X'X)^{-1} X'
X_aug = np.column_stack([np.ones(len(X)), X.to_numpy()])
H_diag = np.einsum('ij,ji->i', X_aug @ np.linalg.inv(X_aug.T @ X_aug), X_aug.T)
axes[1, 1].scatter(H_diag, std_resid, s=8, alpha=0.6)
axes[1, 1].axhline(0, color='C3')
axes[1, 1].set_title('Residuals vs Leverage')
axes[1, 1].set_xlabel('leverage'); axes[1, 1].set_ylabel('std residual')
plt.tight_layout(); plt.show()


**Cook's distance** roughly combines leverage and residual size. Any
point in the upper-right of the leverage plot deserves a look.

## 3. Qualitative predictors and interactions

### Carseats: one-hot the categorical predictors

In [ ]:
print(carseats.dtypes)
print(carseats[['ShelveLoc', 'Urban', 'US']].head())


In [ ]:
df = pd.get_dummies(carseats, columns=['ShelveLoc', 'Urban', 'US'],
                    drop_first=True, dtype=float)
X = df.drop(columns=['Sales'])
y = df['Sales']
m = LinearRegression().fit(X, y)
print(pd.Series(m.coef_, index=X.columns).round(3).to_string())
print(f'R^2 = {m.score(X, y):.4f}')


### Add an interaction: Price × Urban

In [ ]:
df['Price_x_Urban'] = df['Price'] * df['Urban_Yes']
X2 = df.drop(columns=['Sales'])
m2 = LinearRegression().fit(X2, y)
print('coef on Price:           ', m2.coef_[X2.columns.get_loc('Price')].round(4))
print('coef on Price_x_Urban:   ', m2.coef_[X2.columns.get_loc('Price_x_Urban')].round(4))
print(f'R^2 with interaction:    {m2.score(X2, y):.4f}')


### Recap
- Multiple regression in matrix form: $\hat\beta = (X^\top X)^{-1} X^\top y$.
- Always check VIF — collinearity inflates standard errors and flips signs.
- The 2×2 diagnostic panel is your first reflex when a model 'looks fine'.
- One-hot encoding turns categoricals into numeric. Interactions are just
  products of two predictors.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** Load the `penguins` dataset, drop NaN, and fit a simple
linear regression of `body_mass_g` on `flipper_length_mm`. Report the
slope, intercept, and R².

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression
df = load_dataset('penguins').dropna()
X = df[['flipper_length_mm']]
y = df['body_mass_g']
m = LinearRegression().fit(X, y)
print(f'intercept = {m.intercept_:.1f}')
print(f'slope     = {m.coef_[0]:.2f} g per mm')
print(f'R^2       = {m.score(X, y):.4f}')


---

## Exercise 2

**Task 2.** Plot residuals vs fitted for that model. Do the residuals
look like a band of noise around zero, or is there a pattern?

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
import matplotlib.pyplot as plt
y_hat = m.predict(X)
resid = y - y_hat
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(y_hat, resid, s=10, alpha=0.6)
ax.axhline(0, color='C3')
ax.set_xlabel('fitted'); ax.set_ylabel('residual')
plt.show()
# residuals look reasonable — mostly a band, mild fanning out for large fits


---

## Exercise 3

**Task 3.** Bootstrap a 95% confidence interval for the slope. Use
2000 resamples.

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
rng = np.random.default_rng(0)
x = df['flipper_length_mm'].to_numpy()
y = df['body_mass_g'].to_numpy()
n = len(x)
slopes = np.empty(2000)
for k in range(2000):
    idx = rng.choice(n, n, replace=True)
    xb, yb = x[idx], y[idx]
    slopes[k] = ((xb - xb.mean()) * (yb - yb.mean())).sum() / ((xb - xb.mean())**2).sum()
lo, hi = np.quantile(slopes, [0.025, 0.975])
print(f'95% CI for slope: [{lo:.2f}, {hi:.2f}] g/mm')


---

## Exercise 4

**Task 1.** Load `penguins`, drop NaN, one-hot `species` and `sex`,
and fit `body_mass_g ~ all other numeric + dummies`. Print R².

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
from sklearn.linear_model import LinearRegression
# your code here


### Exercise 4 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
from sklearn.linear_model import LinearRegression
df = load_dataset('penguins').dropna()
df = pd.get_dummies(df, columns=['species', 'sex', 'island'], drop_first=True, dtype=float)
X = df.drop(columns=['body_mass_g'])
y = df['body_mass_g']
m = LinearRegression().fit(X, y)
print(f'R^2 = {m.score(X, y):.4f}')


---

## Exercise 5

**Task 2.** Compute VIF for the numeric predictors (`bill_length_mm`,
`bill_depth_mm`, `flipper_length_mm`). Any of them above 5?

In [ ]:
# your code here


### Exercise 5 — Solution

In [ ]:
numeric = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']
Xn = df[numeric]
for col in numeric:
    others = Xn.drop(columns=[col])
    r2 = LinearRegression().fit(others, Xn[col]).score(others, Xn[col])
    print(f'{col:22s}  VIF = {1/(1-r2):.2f}')


---

## Exercise 6

**Task 3.** Add an interaction `flipper_length_mm × species_Chinstrap`
to the model. Does R² improve?

In [ ]:
# your code here


### Exercise 6 — Solution

In [ ]:
df['flipper_x_chinstrap'] = df['flipper_length_mm'] * df['species_Chinstrap']
X2 = df.drop(columns=['body_mass_g'])
m2 = LinearRegression().fit(X2, df['body_mass_g'])
print(f'R^2 with interaction = {m2.score(X2, df["body_mass_g"]):.4f}')
